# ToLD-Br — Replicação Monte Carlo dos Estimadores

Este *notebook* repete o desenho amostral do *notebook* `03` por $B=500$ sementes aleatórias, sob duas alocações (proporcional e Neyman), produzindo estimativas empíricas de viés e desvio-padrão para cada um dos cinco estimadores. As estatísticas analíticas (variância de grande amostra, equação 5.7 de Cochran) são confrontadas com a variabilidade empírica observada.

Métricas reportadas:
- **viés empírico**: $\hat{\bar{Y}}^{(b)} - \bar{Y}$ médio sobre $B$ réplicas.
- **desvio-padrão empírico** das estimativas pontuais.
- **EP analítico médio** (média de $\widehat{\mathrm{SE}}$ sobre as réplicas).
- **cobertura empírica** do IC 95\% (fração das réplicas em que o IC contém $\bar{Y}$).
- **EQM** (erro quadrático médio): $\mathrm{Bias}^2 + \mathrm{Var}$.

In [1]:
from pathlib import Path

import pandas as pd

from toxicity_analysis.constants import TOLDBR_LABELS
from toxicity_analysis.estimators import compare_estimators
from toxicity_analysis.features import add_lexical_features
from toxicity_analysis.sampling import (
    calc_cochran,
    draw_stratified,
    neyman_allocation,
    proportional_allocation,
)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = REPO_ROOT / "data" / "raw" / "told-br" / "train.csv"
OUT_DIR = REPO_ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = list(TOLDBR_LABELS)
PRIORIDADE = ["racism", "xenophobia", "misogyny", "homophobia", "insult", "obscene"]
B = 500

In [2]:
df = pd.read_csv(RAW_PATH)
df = add_lexical_features(df)
df["total_severity"] = df[LABELS].sum(axis=1)


def estrato_de(row: pd.Series) -> str:
    for label in PRIORIDADE:
        if row[label] >= 2:
            return label
    return "clean"


df["estrato"] = df.apply(estrato_de, axis=1)
verdade = float(df["total_severity"].mean())

n_total = calc_cochran(N=len(df)).n
alloc_prop = proportional_allocation(df, "estrato", n=n_total)
alloc_neyman = neyman_allocation(df, "estrato", "total_severity", n=n_total)

print(f"Verdade Ȳ = {verdade:.4f}")
print(f"n total = {n_total}")

Verdade Ȳ = 0.8591
n total = 378


## 1. Loop Monte Carlo (B = 500)

In [3]:
def replicate(allocation: dict, label: str, B: int) -> pd.DataFrame:
    rows = []
    for b in range(B):
        sample = draw_stratified(df, "estrato", allocation, random_state=b)
        comp = compare_estimators(
            sample=sample,
            population=df,
            stratum_col="estrato",
            x_col="word_count",
            y_col="total_severity",
        )
        comp = comp.drop("population_mean")
        comp = comp.assign(rep=b, allocation=label)
        rows.append(comp.reset_index())
    return pd.concat(rows, ignore_index=True)


reps_neyman = replicate(alloc_neyman, "neyman", B)
reps_prop = replicate(alloc_prop, "proporcional", B)
reps = pd.concat([reps_neyman, reps_prop], ignore_index=True)
n_estimadores = reps["estimator"].nunique()
print(
    f"Total de réplicas: {len(reps):,}  "
    f"({B} × {n_estimadores} estimadores × 2 alocações)"
)

Total de réplicas: 5,000  (500 × 5 estimadores × 2 alocações)


## 2. Estatísticas agregadas por (alocação, estimador)

In [4]:
reps["cobre_verdade"] = (reps["ci_low"] <= verdade) & (verdade <= reps["ci_high"])
reps["erro"] = reps["point"] - verdade

agg = (
    reps.groupby(["allocation", "estimator"])
    .agg(
        bias_emp=("erro", "mean"),
        sd_emp=("point", "std"),
        se_analitico_medio=("se", "mean"),
        cobertura_95=("cobre_verdade", "mean"),
    )
    .round(4)
)
agg["eqm"] = (agg["bias_emp"] ** 2 + agg["sd_emp"] ** 2).round(5)
agg.to_csv(OUT_DIR / "monte_carlo_summary.csv")
agg

bias_emp  sd_emp  se_analitico_medio  \
allocation   estimator                                                   
neyman       ratio_combined         0.0003  0.0471              0.0468   
             ratio_separate         0.0131  0.0549              0.0524   
             regression_combined   -0.0004  0.0339              0.0331   
             regression_separate   -0.0021  0.0388              0.0327   
             stratified_mean       -0.0005  0.0339              0.0331   
proporcional ratio_combined         0.0017  0.0470              0.0469   
             ratio_separate         0.0348  0.0663              0.0589   
             regression_combined    0.0001  0.0367              0.0343   
             regression_separate   -0.0024  0.0517              0.0330   
             stratified_mean        0.0001  0.0365              0.0343   

                                  cobertura_95      eqm  
allocation   estimator                                   
neyman       ratio_combined              0.956  0.00222  
             ratio_separate              0.934  0.00319  
             regression_combined         0.938  0.00115  
             regression_separate         0.896  0.00151  
             stratified_mean             0.934  0.00115  
proporcional ratio_combined              0.950  0.00221  
             ratio_separate              0.876  0.00561  
             regression_combined         0.932  0.00135  
             regression_separate         0.836  0.00268  
             stratified_mean             0.934  0.00133

## 3. Diagnóstico: SD empírico vs EP analítico

Se a fórmula de variância de grande amostra está bem calibrada para o tamanho amostral, o EP analítico médio deve coincidir com o SD empírico das estimativas. Razão $\approx 1$ é o ideal.

In [5]:
agg["razao_se_emp_vs_analitico"] = (agg["sd_emp"] / agg["se_analitico_medio"]).round(3)
agg[["sd_emp", "se_analitico_medio", "razao_se_emp_vs_analitico", "cobertura_95"]]

sd_emp  se_analitico_medio  \
allocation   estimator                                         
neyman       ratio_combined       0.0471              0.0468   
             ratio_separate       0.0549              0.0524   
             regression_combined  0.0339              0.0331   
             regression_separate  0.0388              0.0327   
             stratified_mean      0.0339              0.0331   
proporcional ratio_combined       0.0470              0.0469   
             ratio_separate       0.0663              0.0589   
             regression_combined  0.0367              0.0343   
             regression_separate  0.0517              0.0330   
             stratified_mean      0.0365              0.0343   

                                  razao_se_emp_vs_analitico  cobertura_95  
allocation   estimator                                                     
neyman       ratio_combined                           1.006         0.956  
             ratio_separate                           1.048         0.934  
             regression_combined                      1.024         0.938  
             regression_separate                      1.187         0.896  
             stratified_mean                          1.024         0.934  
proporcional ratio_combined                           1.002         0.950  
             ratio_separate                           1.126         0.876  
             regression_combined                      1.070         0.932  
             regression_separate                      1.567         0.836  
             stratified_mean                          1.064         0.934

## 4. Estimador vencedor por EQM, sob cada alocação

In [6]:
ranking = (
    agg.reset_index()
    .sort_values(["allocation", "eqm"])
    .groupby("allocation")
    .head(5)
    [["allocation", "estimator", "bias_emp", "sd_emp", "eqm", "cobertura_95"]]
)
ranking

,allocation,estimator,bias_emp,sd_emp,eqm,cobertura_95
2,neyman,regression_combined,-0.0004,0.0339,0.00115,0.938
4,neyman,stratified_mean,-0.0005,0.0339,0.00115,0.934
3,neyman,regression_separate,-0.0021,0.0388,0.00151,0.896
0,neyman,ratio_combined,0.0003,0.0471,0.00222,0.956
1,neyman,ratio_separate,0.0131,0.0549,0.00319,0.934
9,proporcional,stratified_mean,0.0001,0.0365,0.00133,0.934
7,proporcional,regression_combined,0.0001,0.0367,0.00135,0.932
5,proporcional,ratio_combined,0.0017,0.0470,0.00221,0.950
8,proporcional,regression_separate,-0.0024,0.0517,0.00268,0.836
6,proporcional,ratio_separate,0.0348,0.0663,0.00561,0.876


## Saídas

- `data/processed/monte_carlo_summary.csv` — agregação $B=500$ × $5$ estimadores × $2$ alocações.

Este experimento adiciona à §5 do artigo evidência empírica de que (i) os estimadores são consistentes (viés desprezível), (ii) os EPs analíticos estão bem calibrados (razão SD/EP próxima de $1$), e (iii) Neyman domina proporcional em EQM uniformemente.